# Search & Query: Vector Search 2.0 Collections

This notebook demonstrates querying and searching the Reddit and Zoom collections created in steps 4–5. Covers:

- **Query**: Filter DataObjects by field values (analogous to SQL `WHERE`)
- **Semantic Search**: Natural language queries using auto-generated embeddings
- **Text Search**: Keyword-based full-text search
- **Hybrid Search**: Combine semantic + text with [Reciprocal Rank Fusion (RRF)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- **RRF Weight Tuning**: How adjusting weights shifts result ranking

In [ ]:
import pandas as pd
from google.cloud import vectorsearch_v1beta
from config import PROJECT_ID, REGION, VS_COLLECTION_REDDIT, VS_COLLECTION_ZOOM

# Three specialized SDK clients
vs_client = vectorsearch_v1beta.VectorSearchServiceClient()
do_client = vectorsearch_v1beta.DataObjectServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

# Collection parent paths
REDDIT = f'projects/{PROJECT_ID}/locations/{REGION}/collections/{VS_COLLECTION_REDDIT}'
ZOOM = f'projects/{PROJECT_ID}/locations/{REGION}/collections/{VS_COLLECTION_ZOOM}'

print(f'Reddit: {VS_COLLECTION_REDDIT}')
print(f'Zoom:   {VS_COLLECTION_ZOOM}')

In [ ]:
def search_df(results, fields, max_text=150):
    """Format search results as a ranked DataFrame."""
    rows = []
    for r in results:
        data = r.data_object.data
        row = {}
        for f in fields:
            val = data.get(f, '')
            if isinstance(val, str) and len(val) > max_text:
                val = val[:max_text] + '...'
            elif hasattr(val, '__iter__') and not isinstance(val, str):
                val = ', '.join(str(v) for v in val)
            row[f] = val
        rows.append(row)
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = 'rank'
    return df


def query_df(results, fields, max_text=150):
    """Format query results as a DataFrame."""
    rows = []
    for r in results:
        row = {}
        for f in fields:
            val = r.data.get(f, '')
            if isinstance(val, str) and len(val) > max_text:
                val = val[:max_text] + '...'
            elif hasattr(val, '__iter__') and not isinstance(val, str):
                val = ', '.join(str(v) for v in val)
            row[f] = val
        rows.append(row)
    return pd.DataFrame(rows)


def hybrid_search(parent, query, weights, top_k=10, fields=None):
    """Run hybrid semantic + text search with RRF reranking."""
    if fields is None:
        fields = ['chunk_id', 'text_content']
    output = vectorsearch_v1beta.OutputFields(data_fields=fields)
    request = vectorsearch_v1beta.BatchSearchDataObjectsRequest(
        parent=parent,
        searches=[
            vectorsearch_v1beta.Search(
                semantic_search=vectorsearch_v1beta.SemanticSearch(
                    search_text=query,
                    search_field='text_content_embedding',
                    task_type='QUESTION_ANSWERING',
                    top_k=top_k,
                    output_fields=output,
                )
            ),
            vectorsearch_v1beta.Search(
                text_search=vectorsearch_v1beta.TextSearch(
                    search_text=query,
                    data_field_names=['text_content'],
                    top_k=top_k,
                    output_fields=output,
                )
            ),
        ],
        combine=vectorsearch_v1beta.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch_v1beta.Ranker(
                rrf=vectorsearch_v1beta.ReciprocalRankFusion(weights=weights)
            )
        ),
    )
    response = search_client.batch_search_data_objects(request)
    if response.results:
        return list(response.results[0].results)
    return []

---
# Reddit Collection

The Reddit collection contains flattened thread chunks with conversational context (`THREAD: title | PARENT: context | COMMENT: body`). Each chunk includes metadata: `subreddit`, `karma`, and `is_image_description`.

In [ ]:
# Count DataObjects in the Reddit collection
search_client.aggregate_data_objects(
    vectorsearch_v1beta.AggregateDataObjectsRequest(parent=REDDIT, aggregate='COUNT')
)

## Query: Filtering by Field Values

Query lets you filter DataObjects by their data fields — similar to a SQL `WHERE` clause. Supports comparison operators (`$eq`, `$gt`, `$lt`, `$gte`, `$lte`), logical operators (`$and`, `$or`), and array operators (`$in`, `$nin`).

In [ ]:
# Find high-karma Reddit chunks (quality signal)
results = list(search_client.query_data_objects(
    vectorsearch_v1beta.QueryDataObjectsRequest(
        parent=REDDIT,
        filter={'karma': {'$gt': 50}},
        output_fields=vectorsearch_v1beta.OutputFields(
            data_fields=['chunk_id', 'subreddit', 'karma', 'text_content']
        ),
    )
))
print(f'{len(results)} chunks with karma > 50')
query_df(results, ['chunk_id', 'subreddit', 'karma', 'text_content'])

In [ ]:
# Combined filter: high karma AND image-enriched chunks
results = list(search_client.query_data_objects(
    vectorsearch_v1beta.QueryDataObjectsRequest(
        parent=REDDIT,
        filter={'$and': [{'karma': {'$gte': 20}}, {'is_image_description': {'$eq': True}}]},
        output_fields=vectorsearch_v1beta.OutputFields(
            data_fields=['chunk_id', 'karma', 'is_image_description', 'text_content']
        ),
    )
))
print(f'{len(results)} image-enriched chunks with karma >= 20')
query_df(results, ['chunk_id', 'karma', 'is_image_description', 'text_content'])

## Semantic Search

Semantic search converts your natural language query into an embedding and finds DataObjects with the most similar `text_content_embedding` vectors. The query uses `task_type='QUESTION_ANSWERING'` to pair with the documents indexed as `RETRIEVAL_DOCUMENT` — this asymmetric pairing helps the model understand that queries and documents have different intents.

In [ ]:
query = 'What machine learning methods work best for demand forecasting?'

results = search_client.search_data_objects(
    vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=REDDIT,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field='text_content_embedding',
            task_type='QUESTION_ANSWERING',
            top_k=5,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=['chunk_id', 'subreddit', 'karma', 'text_content']
            ),
        ),
    )
)

print(f'Semantic search: "{query}"\n')
search_df(results, ['chunk_id', 'subreddit', 'karma', 'text_content'])

## Text Search

Text search provides keyword-based full-text matching. Useful for finding specific method names, acronyms, or terms that may not have strong semantic representation in the embedding model.

In [ ]:
query = 'ARIMA'

results = search_client.search_data_objects(
    vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=REDDIT,
        text_search=vectorsearch_v1beta.TextSearch(
            search_text=query,
            data_field_names=['text_content'],
            top_k=5,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=['chunk_id', 'subreddit', 'karma', 'text_content']
            ),
        ),
    )
)

print(f'Text search: "{query}"\n')
search_df(results, ['chunk_id', 'subreddit', 'karma', 'text_content'])

## Hybrid Search with Reciprocal Rank Fusion (RRF)

Hybrid search runs both semantic and text searches in parallel, then combines results using [RRF](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf). RRF merges ranked lists by assigning each result a score based on its rank position in each list, then sorting by combined score.

The `weights` parameter controls the relative importance of each search:
- `[1.0, 1.0]` — equal balance between semantic and text
- `[3.0, 1.0]` — favor semantic understanding (better for intent-based queries)
- `[1.0, 3.0]` — favor keyword matching (better for exact-term queries)

In [ ]:
query = 'seasonal ARIMA forecasting methods'
fields = ['chunk_id', 'subreddit', 'karma', 'text_content']

results = hybrid_search(REDDIT, query, weights=[1.0, 1.0], top_k=5, fields=fields)

print(f'Hybrid search (equal weights [1, 1]): "{query}"\n')
search_df(results, fields)

### RRF Weight Comparison

The table below shows how the same query produces different rankings under three weight configurations. A lower rank number means higher relevance. Watch how chunks shift position as the balance changes between semantic understanding and keyword matching.

In [ ]:
query = 'seasonal ARIMA forecasting methods'
fields = ['chunk_id', 'text_content']

configs = {
    'Equal [1, 1]': [1.0, 1.0],
    'Semantic [3, 1]': [3.0, 1.0],
    'Text [1, 3]': [1.0, 3.0],
}

rankings = {}
for label, weights in configs.items():
    results = hybrid_search(REDDIT, query, weights, top_k=10, fields=fields)
    for rank, r in enumerate(results, 1):
        cid = r.data_object.data['chunk_id']
        if cid not in rankings:
            text = r.data_object.data['text_content']
            rankings[cid] = {'chunk_id': cid, 'text_preview': text[:100] + '...'}
        rankings[cid][label] = rank

comparison = pd.DataFrame(rankings.values()).sort_values('Equal [1, 1]')
print(f'RRF weight comparison: "{query}"')
print('Lower rank = higher relevance. Watch how chunks shift position.\n')
comparison

---
# Zoom Collection

The Zoom collection contains sliding-window transcript chunks with speaker labels and timestamp ranges. Each chunk covers ~15 cues with 5-cue overlap between neighbors. Non-first windows include a Gemini-generated rolling summary prefix.

In [ ]:
# Count DataObjects in the Zoom collection
search_client.aggregate_data_objects(
    vectorsearch_v1beta.AggregateDataObjectsRequest(parent=ZOOM, aggregate='COUNT')
)

## Query: Browsing Transcript Chunks

Filter by timestamp to find chunks from later in meetings, or browse all chunks to see the speaker distribution.

In [ ]:
# Find transcript chunks from later in meetings (past 5 minutes)
results = list(search_client.query_data_objects(
    vectorsearch_v1beta.QueryDataObjectsRequest(
        parent=ZOOM,
        filter={'timestamp_start': {'$gt': 300}},
        output_fields=vectorsearch_v1beta.OutputFields(
            data_fields=['chunk_id', 'speaker_list', 'timestamp_start', 'timestamp_end', 'text_content']
        ),
    )
))
print(f'{len(results)} chunks starting after 5:00')
query_df(results, ['chunk_id', 'speaker_list', 'timestamp_start', 'timestamp_end', 'text_content'])

## Semantic Search

In [ ]:
query = 'How should we evaluate forecast model accuracy and performance?'

results = search_client.search_data_objects(
    vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=ZOOM,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field='text_content_embedding',
            task_type='QUESTION_ANSWERING',
            top_k=5,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=['chunk_id', 'speaker_list', 'text_content']
            ),
        ),
    )
)

print(f'Semantic search: "{query}"\n')
search_df(results, ['chunk_id', 'speaker_list', 'text_content'])

## Text Search

In [ ]:
query = 'Prophet'

results = search_client.search_data_objects(
    vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=ZOOM,
        text_search=vectorsearch_v1beta.TextSearch(
            search_text=query,
            data_field_names=['text_content'],
            top_k=5,
            output_fields=vectorsearch_v1beta.OutputFields(
                data_fields=['chunk_id', 'speaker_list', 'text_content']
            ),
        ),
    )
)

print(f'Text search: "{query}"\n')
search_df(results, ['chunk_id', 'speaker_list', 'text_content'])

## Hybrid Search with RRF Weight Comparison

In [ ]:
query = 'seasonal patterns and Prophet model'
fields = ['chunk_id', 'text_content']

configs = {
    'Equal [1, 1]': [1.0, 1.0],
    'Semantic [3, 1]': [3.0, 1.0],
    'Text [1, 3]': [1.0, 3.0],
}

rankings = {}
for label, weights in configs.items():
    results = hybrid_search(ZOOM, query, weights, top_k=10, fields=fields)
    for rank, r in enumerate(results, 1):
        cid = r.data_object.data['chunk_id']
        if cid not in rankings:
            text = r.data_object.data['text_content']
            rankings[cid] = {'chunk_id': cid, 'text_preview': text[:100] + '...'}
        rankings[cid][label] = rank

comparison = pd.DataFrame(rankings.values()).sort_values('Equal [1, 1]')
print(f'RRF weight comparison: "{query}"')
print('Lower rank = higher relevance.\n')
comparison

---
# Key Takeaways

| Search Type | Strengths | Weaknesses | Best For |
|-------------|-----------|------------|----------|
| **Semantic** | Understands intent, finds synonyms | May miss exact keywords | Natural language questions |
| **Text** | Precise keyword matching | No semantic understanding | Specific terms, acronyms, names |
| **Hybrid (RRF)** | Best of both | Requires weight tuning | Production search systems |

**RRF weight tuning tips:**
- Start with equal weights `[1, 1]` as a baseline
- Increase semantic weight when users ask natural language questions
- Increase text weight when users search for specific terms or codes
- The comparison tables above show how the same query can surface different chunks depending on the weight balance — chunks containing exact keyword matches rise when text weight increases, while chunks with conceptually related content rise when semantic weight increases